Import all relevant functions.

In [ ]:
import pandas as pd
import pandasql as ps 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from joblib import dump, load
import sqlite3

Store the cleaned dataset into a variable.

In [ ]:
dataset_path = "assets/data.db"
with sqlite3.connect(dataset_path) as conn:
    dataset = pd.read_sql("SELECT * FROM data", conn)

dataset

Use one-hot encoding to convert all variables to numerical form.

In [ ]:
dataset_encoded = pd.get_dummies(dataset)
dataset_encoded = dataset_encoded.astype(int)
dataset_encoded

Seperate the bot column from the rest of the columns.

In [ ]:
y = dataset_encoded["bot"]
X = dataset_encoded.drop(["bot"], axis=1)

In [ ]:
X

In [ ]:
y

Split X and y into training and testing datasets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
y_train

In [ ]:
y_test

Scale the features on training data and apply it to both datasets.

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled)
X_test_scaled

Train with the classifier using the KNN with the highest accuracy.

In [ ]:
acc_array = []
acc_highest = 0
acc_neighbour = 0

for i in range(len(y_train)):
    knn_class = KNeighborsClassifier(n_neighbors=i+1)
    knn_class.fit(X_train_scaled, y_train)
    
    y_pred = knn_class.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)

    acc_array.append(accuracy)

for i in range(len(acc_array)):
    if acc_array[i] > acc_highest:
        acc_highest = acc_array[i]
        acc_neighbour = i+1

knn_class = KNeighborsClassifier(n_neighbors=acc_neighbour)
knn_class.fit(X_train_scaled, y_train)

Test out the accuracy of the classifier along with the KNN value used.

In [ ]:
y_pred = knn_class.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred) 
print(f"Accuracy : {accuracy:.4f}")
print(f"KNN Value : {acc_neighbour}")

In [ ]:
acc_array = []
acc_highest = 0
acc_neighbour = 0

for i in range(len(y_train)):
    knn_class = KNeighborsClassifier(n_neighbors=i+1)
    knn_class.fit(X_train_scaled, y_train)
    
    y_pred = knn_class.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)

    acc_array.append(accuracy)

for i in range(len(acc_array)):
    if acc_array[i] > acc_highest:
        acc_highest = acc_array[i]
        acc_neighbour = i+1

knn_class = KNeighborsClassifier(n_neighbors=acc_neighbour)
knn_class.fit(X_train_scaled, y_train)
    
y_pred = knn_class.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred) 
print(accuracy)
print(acc_neighbour)

Checks if accuracy improves if a feature is removed and outputs the features that improve or worsen accuracy when removed.

In [ ]:
worse_removed = []
better_removed = []
remove = []

for i in range(len(dataset.columns)-1):
    tdataset = dataset.drop(dataset.columns[i], axis=1)
    tdataset_encoded = pd.get_dummies(tdataset)

    ty = tdataset_encoded["bot"]
    tX = tdataset_encoded.drop(["bot"], axis=1)

    tX_train, tX_test, ty_train, ty_test = train_test_split(tX, ty, test_size=0.2, random_state=42)

    tscaler = StandardScaler()
    tscaler.fit(tX_train)

    tX_train_scaled = tscaler.transform(tX_train)
    tX_test_scaled = tscaler.transform(tX_test)

    tknn_class = KNeighborsClassifier(n_neighbors=18)
    tknn_class.fit(tX_train_scaled, ty_train)

    ty_pred = tknn_class.predict(tX_test_scaled)

    if accuracy_score(ty_test, ty_pred) <= 0.9692307692307692:
        worse_removed.append(dataset.columns[i])
    else:
        better_removed.append(dataset.columns[i])
        remove.append(i)

print(worse_removed)
print(better_removed)

In [ ]:
print(type(scaler), type(knn_class))
dump(knn_class, "assets/bvh_knn_model48.joblib")
dump(scaler, "assets/bvh_scaler48.joblib")
dump(list(X.columns), "assets/bvh_features48.joblib")